# Práctica 2 — Números (pseudo)aleatorios y Generación de Variables Aleatorias
## Secciones 5.2, 5.3 y 5.4

**Curso:** Simulación Computacional — Universidad de los Llanos
**Código de formato:** FO-DOC-112

En este notebook se desarrolla y explican los tres puntos de la practica de laboratorio:

- **5.2.** uso de números aleatorios para aproximar el valor de varias integrales, tal como se pide en los ejercicios 3 a 9 del capítulo 3 del libro de Ross.
- **5.3.** generación de variables aleatorias discretas usando el método de la transformada inversa (y también el método de composición), a partir de un generador de números pseudoaleatorios de tipo congruencial mixto indicado en la guía.
- **5.4.** repetición de la simulación de una fila de un solo servidor que se hizo en la Práctica 1, pero cambiando la forma de generar los tiempos entre llegadas y los tiempos de servicio: en vez de usar dados o ruletas, se construyen generadores propios de una variable Poisson y de una variable Binomial.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')


---
# 5.2. Uso de números aleatorios para evaluar integrales

se entiende que la idea del método de Monte Carlo para aproximar una integral es la siguiente: en vez de resolver la integral de forma analítica, se aprovecha el hecho de que una integral definida se puede interpretar como un promedio de la función evaluada en puntos aleatorios, multiplicado por el tamaño del intervalo sobre el que se integra.

en términos simples, el algoritmo que se implementa hace lo siguiente:

1. se genera una gran cantidad de números pseudoaleatorios uniformes dentro del intervalo donde está definida la integral.
2. se evalúa la función que se quiere integrar en cada uno de esos números.
3. se calcula el promedio de todos esos valores.
4. ese promedio se multiplica por la longitud del intervalo, y el resultado es la aproximación de la integral.

cuando la integral es doble, el mismo razonamiento se extiende generando pares de números aleatorios (uno para cada variable) y promediando la función evaluada en esos pares, multiplicando luego por el área de la región sobre la que se integra.

para las integrales cuyos límites llegan hasta el infinito, se entiende que no es posible generar directamente números aleatorios "hasta el infinito", así que se usa un cambio de variable que transforma ese intervalo infinito en el intervalo de 0 a 1. De esa manera, sí se pueden generar números uniformes normales, pero antes de promediar la función original, hay que dividir cada valor por un factor de corrección que aparece por el cambio de variable. Cuando la función que se integra es simétrica (par), basta con hacer el cálculo solo del lado positivo y luego multiplicar el resultado por dos, en lugar de trabajar con todo el eje.

en todos los casos se usa una cantidad grande de números pseudoaleatorios (100.000) en cada estimación, ya que entre más números se usen, más cerca estará el promedio del valor real de la integral.


In [ ]:
def mc_integral_1d(g, a, b, n=100_000, rng=None):
    """Estima integral_a^b g(x) dx por Monte Carlo simple."""
    rng = rng or np.random.default_rng(2026)
    u = rng.uniform(a, b, n)
    return (b - a) * np.mean(g(u))

def mc_integral_0_inf(h, n=100_000, rng=None):
    """Estima integral_0^inf h(x) dx via sustitucion y=1/(1+x), x=(1-y)/y, dx=dy/y^2."""
    rng = rng or np.random.default_rng(2026)
    y = rng.uniform(0, 1, n)
    x = (1 - y) / y
    integrand = h(x) / y**2
    return np.mean(integrand)

rng_global = np.random.default_rng(2026)


## Ejercicio 3

se debe aproximar, usando números aleatorios, el valor de la integral definida entre 0 y 1 de la función que consiste en elevar el número e al exponente e elevado a x. Es decir, primero se calcula e elevado a x, y ese resultado se usa nuevamente como exponente de e.


In [ ]:
g3 = lambda x: np.exp(np.exp(x))
theta3 = mc_integral_1d(g3, 0, 1, rng=rng_global)
print(f"Estimacion Monte Carlo (Ejercicio 3): {theta3:.6f}")


Estimacion Monte Carlo (Ejercicio 3): 6.299839


## Ejercicio 4

se debe aproximar el valor de la integral definida entre 0 y 1 de la función que resulta de tomar uno menos x al cuadrado, y elevar ese resultado a la potencia tres medios (es decir, la raíz cuadrada de su cubo).


In [ ]:
g4 = lambda x: (1 - x**2)**1.5
theta4 = mc_integral_1d(g4, 0, 1, rng=rng_global)
print(f"Estimacion Monte Carlo (Ejercicio 4): {theta4:.6f}")
# Valor analitico de referencia: 3*pi/16
print(f"Valor analitico 3*pi/16          : {3*np.pi/16:.6f}")


Estimacion Monte Carlo (Ejercicio 4): 0.590955
Valor analitico 3*pi/16          : 0.589049


## Ejercicio 5

hay que aproximar el valor de la integral definida entre menos 2 y 2 de la función exponencial de x más x al cuadrado.


In [ ]:
g5 = lambda x: np.exp(x + x**2)
theta5 = mc_integral_1d(g5, -2, 2, rng=rng_global)
print(f"Estimacion Monte Carlo (Ejercicio 5): {theta5:.6f}")


Estimacion Monte Carlo (Ejercicio 5): 92.978184


## Ejercicio 6

hay aproximar el valor de la integral, definida entre 0 e infinito, de la función x dividido entre uno más x al cuadrado, todo elevado al cuadrado.

como el límite superior es infinito, se aplica el cambio de variable explicado en la introducción de la sección 5.2 para poder trabajar con números aleatorios uniformes normales entre 0 y 1.


In [ ]:
h6 = lambda x: x * (1 + x**2)**(-2)
theta6 = mc_integral_0_inf(h6, rng=rng_global)
print(f"Estimacion Monte Carlo (Ejercicio 6): {theta6:.6f}")
# Valor analitico de referencia: 1/2
print(f"Valor analitico 1/2               : {0.5:.6f}")


Estimacion Monte Carlo (Ejercicio 6): 0.500928
Valor analitico 1/2               : 0.500000


## Ejercicio 7
se debe aproximar el valor de la integral, definida entre menos infinito y más infinito, de la función exponencial de menos x al cuadrado.

como esta función es simétrica respecto al eje vertical (es una función par), el estudiante aprovecha que el resultado sobre todo el eje real es equivalente a calcular dos veces la integral solo desde 0 hasta infinito. Por eso se reutiliza el mismo cambio de variable para límite infinito que se usó en el ejercicio anterior.


In [ ]:
h7 = lambda x: np.exp(-x**2)
theta7 = 2 * mc_integral_0_inf(h7, rng=rng_global)
print(f"Estimacion Monte Carlo (Ejercicio 7): {theta7:.6f}")
# Valor analitico de referencia: sqrt(pi)
print(f"Valor analitico sqrt(pi)          : {np.sqrt(np.pi):.6f}")


Estimacion Monte Carlo (Ejercicio 7): 1.767204
Valor analitico sqrt(pi)          : 1.772454


## Ejercicio 8

se debe aproximar el valor de una integral doble, definida sobre el cuadrado que va de 0 a 1 tanto en x como en y, de la función exponencial de la suma de x más y, elevada al cuadrado.

para resolverla se generan parejas de números aleatorios uniformes independientes, uno para representar a x y otro para representar a y, se evalúa la función con esa pareja de valores, y luego se promedian todos los resultados obtenidos.


In [ ]:
def mc_integral_2d_unit_square(g, n=200_000, rng=None):
    rng = rng or np.random.default_rng(2026)
    u = rng.uniform(0, 1, n)
    v = rng.uniform(0, 1, n)
    return np.mean(g(u, v))

g8 = lambda x, y: np.exp((x + y)**2)
theta8 = mc_integral_2d_unit_square(g8, rng=rng_global)
print(f"Estimacion Monte Carlo (Ejercicio 8): {theta8:.6f}")


Estimacion Monte Carlo (Ejercicio 8): 4.902697


## Ejercicio 9

 se aproxima el valor de una integral doble un poco más particular: x va de 0 a infinito, y para cada x, la variable y va de 0 hasta ese mismo valor de x, de la función exponencial de menos la suma de x más y.

siguiendo la sugerencia del enunciado, el estudiante entiende que en vez de manejar directamente ese límite variable (donde y depende de x), es más fácil definir un indicador que valga 1 cuando y es menor que x, y valga 0 en caso contrario. De esta forma la integral se puede reescribir como una integral sobre todo el primer cuadrante (x e y entre 0 e infinito, cada una por su lado), de la misma función exponencial multiplicada por ese indicador.

con esa reformulación, se generan x y y de manera independiente usando el cambio de variable para límite infinito, se evalúa la función junto con el indicador, y se promedia el resultado.


In [ ]:
def mc_integral_quadrant_infinite(g, n=200_000, rng=None):
    """Estima integral_0^inf integral_0^inf g(x,y) dy dx usando la sustitucion
    y1=1/(1+x), y2=1/(1+y) por separado (x=(1-y1)/y1, y=(1-y2)/y2)."""
    rng = rng or np.random.default_rng(2026)
    y1 = rng.uniform(0, 1, n)
    y2 = rng.uniform(0, 1, n)
    x = (1 - y1) / y1
    y = (1 - y2) / y2
    integrand = g(x, y) / (y1**2 * y2**2)
    return np.mean(integrand)

g9 = lambda x, y: np.exp(-(x + y)) * (y < x)
theta9 = mc_integral_quadrant_infinite(g9, rng=rng_global)
print(f"Estimacion Monte Carlo (Ejercicio 9): {theta9:.6f}")
# Valor analitico de referencia: 1/2 * integral_0^inf e^{-2x}(1-...) -> valor cerrado = 1/4 aprox? calculado numericamente abajo


Estimacion Monte Carlo (Ejercicio 9): 0.500137


Se resumen los resultados de la sección 5.2 en la siguiente tabla, para que se pueda comparar de un vistazo el valor aproximado que arrojó cada uno de los ejercicios anteriores:


In [ ]:
resumen_52 = pd.DataFrame({
    'Ejercicio': [3, 4, 5, 6, 7, 8, 9],
    'Integral': [
        'Integral 0-1 exp(e^x) dx',
        'Integral 0-1 (1-x^2)^(3/2) dx',
        'Integral -2-2 e^(x+x^2) dx',
        'Integral 0-inf x(1+x^2)^-2 dx',
        'Integral -inf-inf e^(-x^2) dx',
        'Integral 0-1 0-1 e^((x+y)^2) dy dx',
        'Integral 0-inf 0-x e^-(x+y) dy dx',
    ],
    'Estimacion Monte Carlo': [theta3, theta4, theta5, theta6, theta7, theta8, theta9]
})
resumen_52


,Ejercicio,Integral,Estimacion Monte Carlo
0,3,Integral 0-1 exp(e^x) dx,6.299839
1,4,Integral 0-1 (1-x^2)^(3/2) dx,0.590955
2,5,Integral -2-2 e^(x+x^2) dx,92.978184
3,6,Integral 0-inf x(1+x^2)^-2 dx,0.500928
4,7,Integral -inf-inf e^(-x^2) dx,1.767204
5,8,Integral 0-1 0-1 e^((x+y)^2) dy dx,4.902697
6,9,Integral 0-inf 0-x e^-(x+y) dy dx,0.500137


---
# 5.3. Generación de Variables Aleatorias Discretas

Antes de generar cualquier variable aleatoria discreta, el estudiante necesita una fuente de números pseudoaleatorios uniformes. Para eso se usa un generador congruencial mixto, cuyo funcionamiento se puede resumir así: se parte de un número inicial (llamado semilla), y cada nuevo número de la secuencia se obtiene multiplicando el número anterior por una constante, sumándole otra constante, y tomando el residuo de dividir ese resultado entre un módulo muy grande (una potencia de 2). Ese residuo, dividido entre el módulo, es el número pseudoaleatorio uniforme entre 0 y 1 que se usa después.

Los parámetros de este generador (la semilla inicial, el multiplicador, el término que se suma y el módulo) son los que indica la guía del laboratorio, y se mantienen fijos durante todo el desarrollo de esta sección para que los resultados sean reproducibles.


In [ ]:
class GLC:
    """Generador congruencial lineal mixto."""
    def __init__(self, x0, a, c, m):
        self.x = x0
        self.a = a
        self.c = c
        self.m = m

    def next_u(self):
        self.x = (self.a * self.x + self.c) % self.m
        return self.x / self.m

    def sample(self, n):
        return np.array([self.next_u() for _ in range(n)])

# Parametros dados en el numeral 5.3
X0, A, C, M = 2391, 25214903917, 11, 2**48


## 5.3.a. Probabilidades: 0.20, 0.15, 0.25 y 0.40

el método de la transformada inversa funciona de la siguiente manera para variables discretas: primero se construye la función de distribución acumulada, es decir, se van sumando las probabilidades una tras otra hasta completar el total. Luego se genera un número aleatorio uniforme entre 0 y 1, y se recorre esa acumulación de probabilidades hasta encontrar el primer valor de la variable para el cual la suma acumulada supera (o iguala) al número generado. Ese valor es el que se devuelve como resultado de la simulación.


In [ ]:
def inversa_discreta(valores, probs, u):
    """Transformada inversa para un vector u de uniformes ya generadas."""
    cum = np.cumsum(probs)
    idx = np.searchsorted(cum, u, side='right')
    idx = np.clip(idx, 0, len(valores) - 1)
    return np.array(valores)[idx]

glc_a = GLC(X0, A, C, M)
u_a = glc_a.sample(100)
valores_a = [1, 2, 3, 4]
probs_a = [0.20, 0.15, 0.25, 0.40]
X_a = inversa_discreta(valores_a, probs_a, u_a)

print("Primeros 20 valores generados:", X_a[:20])
frecuencias_a = pd.Series(X_a).value_counts(normalize=True).sort_index()
print("\nFrecuencias relativas observadas vs. teoricas:")
tabla_a = pd.DataFrame({'p_teorica': probs_a, 'frecuencia_observada': frecuencias_a.values}, index=valores_a)
tabla_a


Primeros 20 valores generados: [2 4 4 4 4 3 4 4 3 4 2 1 4 4 1 1 1 1 2 3]

Frecuencias relativas observadas vs. teoricas:


,p_teorica,frecuencia_observada
1,0.200000,0.210000
2,0.150000,0.140000
3,0.250000,0.270000
4,0.400000,0.380000


## 5.3.b. Probabilidades: P(X=1)=0.30, P(X=2)=0.20, P(X=3)=0.35 y P(X=4)=0.15

Se aplica exactamente la misma lógica de transformada inversa explicada en el punto anterior, únicamente cambiando las probabilidades asociadas a cada valor de la variable.


In [ ]:
glc_b = GLC(X0, A, C, M)
u_b = glc_b.sample(100)
valores_b = [1, 2, 3, 4]
probs_b = [0.30, 0.20, 0.35, 0.15]
X_b = inversa_discreta(valores_b, probs_b, u_b)

print("Primeros 20 valores generados:", X_b[:20])
frecuencias_b = pd.Series(X_b).value_counts(normalize=True).sort_index()
tabla_b = pd.DataFrame({'p_teorica': probs_b, 'frecuencia_observada': frecuencias_b.values}, index=valores_b)
tabla_b


Primeros 20 valores generados: [1 3 3 3 3 2 3 4 2 3 2 1 3 3 1 1 1 1 1 2]


,p_teorica,frecuencia_observada
1,0.300000,0.320000
2,0.200000,0.220000
3,0.350000,0.290000
4,0.150000,0.170000


## 5.3.c. Probabilidades: p1 = 1/3 y p2 = 2/3

En este caso se genera la variable con tres tamaños de muestra distintos (100, 1000 y 10000 valores), y en cada caso calcula qué proporción de los valores generados resultó igual a 1. La idea es observar cómo, entre más valores se generan, esa proporción muestral se acerca cada vez más a la probabilidad teórica de un tercio.


In [ ]:
valores_c = [1, 2]
probs_c = [1/3, 2/3]

resultados_c = []
for n in [100, 1000, 10000]:
    glc_c = GLC(X0, A, C, M)          # se reinicia la semilla para cada tamano de muestra
    u_c = glc_c.sample(n)
    X_c = inversa_discreta(valores_c, probs_c, u_c)
    proporcion_1 = np.mean(X_c == 1)
    resultados_c.append((n, proporcion_1))
    print(f"n = {n:>6}  ->  proporcion de X=1: {proporcion_1:.4f}   (teorico: {1/3:.4f})")

tabla_c = pd.DataFrame(resultados_c, columns=['n', 'proporcion_X_igual_1'])
tabla_c


n =    100  ->  proporcion de X=1: 0.3400   (teorico: 0.3333)
n =   1000  ->  proporcion de X=1: 0.3280   (teorico: 0.3333)
n =  10000  ->  proporcion de X=1: 0.3316   (teorico: 0.3333)


,n,proporcion_X_igual_1
0,100,0.340000
1,1000,0.328000
2,10000,0.331600


**Observación:** Se nota que, a medida que aumenta la cantidad de valores generados, la proporción muestral de valores iguales a 1 se acerca cada vez más al valor teórico de un tercio. Esto es un ejemplo práctico de la Ley de los Grandes Números: entre más repeticiones se hagan de un experimento aleatorio, el promedio observado tiende a parecerse más al valor esperado en teoría.


## 5.3.d. Método de composición

En este punto la variable puede tomar los valores del 1 al 10, con probabilidades 0.06, 0.06, 0.06, 0.06, 0.06, 0.15, 0.13, 0.14, 0.15 y 0.13, respectivamente.

el método de composición se usa cuando la distribución de probabilidad se puede descomponer como una mezcla de dos (o más) distribuciones más sencillas de simular. En este caso, la distribución original se puede ver como una combinación, en una proporción de 60% y 40%, de dos partes:

- Una parte uniforme, donde los diez valores tienen exactamente la misma probabilidad (un décimo cada uno). Esta parte explica, por sí sola, la probabilidad base de 0.06 que comparten todos los valores.
- Una parte adicional que solo afecta a los valores del 6 al 10, y que recoge el excedente de probabilidad que estos cinco valores tienen por encima de ese 0.06 base. Esta segunda parte se normaliza para que sus propias probabilidades sumen 1, y queda concentrada únicamente en los valores del 6 al 10.

El algoritmo de simulación que se deriva de esta descomposición funciona así:

1. Se genera un primer número aleatorio uniforme entre 0 y 1.
2. Si ese número es menor que 0.6, se genera un segundo número aleatorio uniforme y con él se elige, con igual probabilidad, cualquiera de los valores del 1 al 10 (es decir, se simula la parte uniforme).
3. Si ese primer número es mayor o igual a 0.6, se genera un segundo número aleatorio uniforme y, mediante transformada inversa, se elige uno de los valores entre el 6 y el 10, mirando las probabilidades ya normalizadas de esa segunda parte.


In [ ]:
p_d = [0.06, 0.06, 0.06, 0.06, 0.06, 0.15, 0.13, 0.14, 0.15, 0.13]
valores_d = list(range(1, 11))

alpha = 0.6
q_d = [1/10] * 10
r_vals = [6, 7, 8, 9, 10]
r_probs = [(p_d[j-1] - alpha * (1/10)) / (1 - alpha) for j in r_vals]
print("Comprobacion r_j:", np.round(r_probs, 4), " suma =", round(sum(r_probs), 4))

def composicion_X(glc):
    u1 = glc.next_u()
    u2 = glc.next_u()
    if u1 < alpha:
        return int(np.ceil(10 * u2))
    else:
        return int(inversa_discreta(r_vals, r_probs, np.array([u2]))[0])

glc_d = GLC(X0, A, C, M)
X_d = np.array([composicion_X(glc_d) for _ in range(100)])

print("\nPrimeros 20 valores generados:", X_d[:20])
frecuencias_d = pd.Series(X_d).value_counts(normalize=True).sort_index().reindex(valores_d, fill_value=0)
tabla_d = pd.DataFrame({'p_teorica': p_d, 'frecuencia_observada': frecuencias_d.values}, index=valores_d)
tabla_d


Comprobacion r_j: [0.225 0.175 0.2   0.225 0.175]  suma = 1.0

Primeros 20 valores generados: [ 8  9  7 10  8  1  9  2  2  5  8  4 10 10  6  7  7  7  6  9]


,p_teorica,frecuencia_observada
1,0.060000,0.090000
2,0.060000,0.060000
3,0.060000,0.000000
4,0.060000,0.120000
5,0.060000,0.090000
6,0.150000,0.150000
7,0.130000,0.170000
8,0.140000,0.130000
9,0.150000,0.070000
10,0.130000,0.120000


---
# 5.4. Simulación Ad Hoc con generación de variables aleatorias discretas

Se hace la repetició de la simulación de una fila con un solo servidor que ya se había desarrollado en la Práctica 1 (secciones 5.1 y 5.2 de ese laboratorio: 20 clientes, 10 corridas, e intervalos de confianza). La diferencia frente a la Práctica 1 es que ahora los tiempos entre llegadas y los tiempos de servicio ya no se generan lanzando dados o girando una ruleta (es decir, con una distribución uniforme discreta), sino con dos generadores propios que el mismo estudiante construye:

- **Tiempo entre llegadas:** se genera con un algoritmo propio para una variable Poisson con parámetro lambda igual a 10.
- **Tiempo de servicio:** se genera con un algoritmo propio para una variable Binomial con n igual a 10 y probabilidad de éxito igual a 0.40.

La estructura de la tabla de simulación (columnas Customer, Time Between Arrivals, Arrival Time, Service Time, Service Begins, Time Service Ends, Time in System, Idle Time, Time in Queue) y las cinco medidas de desempeño que se calculan al final son las mismas que se usaron en el archivo Ejemplo_fila_banco_20_clientes.xlsx de la Práctica 1, para que los resultados de ambas prácticas se puedan comparar directamente.


## Generadores propios de variables aleatorias (Poisson y Binomial)

**Generador de una variable Poisson:**  Este algoritmo va calculando, paso a paso, la probabilidad de que la variable tome el valor 0, luego 1, luego 2, y así sucesivamente, sumando esas probabilidades una tras otra hasta que la suma acumulada supere un número aleatorio uniforme que se generó al comienzo. El valor de la variable que se devuelve es el número de pasos que se necesitaron para que la suma acumulada alcanzara ese número aleatorio. En la práctica, esto se implementa de forma eficiente porque cada nueva probabilidad se puede calcular a partir de la anterior, multiplicándola por lambda y dividiéndola entre el contador de pasos más uno, en vez de tener que recalcular cada probabilidad desde cero.

**Generador de una variable Binomial:**  este algoritmo sigue la misma idea general (ir acumulando probabilidades hasta superar un número aleatorio uniforme), pero aprovechando que, en una distribución Binomial, la probabilidad de un valor se puede obtener a partir de la probabilidad del valor anterior multiplicándola por un factor que depende de n, de p y del valor actual. Gracias a esa relación entre valores consecutivos, no es necesario calcular cada probabilidad de forma independiente, sino que se construyen unas a partir de otras de manera recursiva, lo cual hace el algoritmo más rápido.


In [ ]:
def poisson_rv(lam, glc):
    """Genera una realizacion Poisson(lam) por el metodo secuencial (Ross 4.2)."""
    p = np.exp(-lam)
    F = p
    X = 0
    U = glc.next_u()
    while U >= F:
        p = lam / (X + 1) * p
        F += p
        X += 1
    return X

def binomial_rv(n, p, glc):
    """Genera una realizacion Binomial(n,p) por transformada inversa recursiva (Ross 4.3)."""
    c = p / (1 - p)
    prob = (1 - p) ** n          # P{X = 0}
    F = prob
    U = glc.next_u()
    i = 0
    while U >= F:
        prob = c * (n - i) / (i + 1) * prob
        F += prob
        i += 1
    return i

# Verificacion rapida: medias empiricas vs teoricas
glc_check = GLC(7919, 25214903917, 11, 2**48)
poiss_sample = [poisson_rv(10, glc_check) for _ in range(20_000)]
binom_sample = [binomial_rv(10, 0.40, glc_check) for _ in range(20_000)]
print(f"Poisson(10):   media empirica = {np.mean(poiss_sample):.4f}  (teorica = 10)")
print(f"Binomial(10,0.4): media empirica = {np.mean(binom_sample):.4f}  (teorica = {10*0.40})")


Poisson(10):   media empirica = 10.0286  (teorica = 10)
Binomial(10,0.4): media empirica = 3.9874  (teorica = 4.0)


## Función de simulación de la cola (mismo esquema de la Práctica 1)

Aqui se reutiliza la misma lógica que se usó en la Práctica 1 para simular la fila: el primer cliente siempre llega en el instante cero, sin necesidad de generar un tiempo entre llegadas para él. Para los clientes siguientes, sí se genera un tiempo entre llegadas (ahora con el generador propio de Poisson) que se va acumulando sobre el tiempo de llegada del cliente anterior, y también se genera un tiempo de servicio (ahora con el generador propio de Binomial) para saber cuánto dura la atención de cada cliente. Con esos dos tiempos, la función calcula, cliente por cliente, en qué momento empieza y termina su servicio, cuánto tiempo pasa en el sistema, cuánto tiempo debe esperar en la fila, y cuánto tiempo queda ocioso el servidor entre un cliente y el siguiente.


In [ ]:
def simular_cola(n_clientes, lam_poisson, n_binom, p_binom, seed):
    glc = GLC((seed * 2654435761 + 12345) % (2**48), A, C, M)

    filas = []
    arrival_time = 0
    service_end_prev = 0

    for cliente in range(1, n_clientes + 1):
        if cliente == 1:
            tba = None
            arrival_time = 0
        else:
            tba = poisson_rv(lam_poisson, glc)
            arrival_time += tba

        service_time = binomial_rv(n_binom, p_binom, glc)
        # evitar tiempos de servicio nulos (serian triviales); Binomial(10,0.4) rara vez da 0
        service_begin = max(arrival_time, service_end_prev)
        service_end = service_begin + service_time
        time_in_system = service_end - arrival_time
        idle_time = max(0, arrival_time - service_end_prev)
        time_in_queue = service_begin - arrival_time

        filas.append({
            'Customer': cliente,
            'Time Between Arrivals': tba if tba is not None else '-',
            'Arrival Time': arrival_time,
            'Service Time': service_time,
            'Service Begins': service_begin,
            'Time Service Ends': service_end,
            'Time in System': time_in_system,
            'Idle Time': idle_time,
            'Time in Queue': time_in_queue,
        })
        service_end_prev = service_end

    df = pd.DataFrame(filas)

    total_time = df['Time Service Ends'].iloc[-1]
    avg_time_system = df['Time in System'].mean()
    pct_idle = df['Idle Time'].sum() / total_time
    avg_wait_per_customer = df['Time in Queue'].mean()
    frac_wait = (df['Time in Queue'] > 0).mean()
    waited = df.loc[df['Time in Queue'] > 0, 'Time in Queue']
    avg_wait_of_waiters = waited.mean() if len(waited) > 0 else 0.0

    medidas = {
        'Tiempo promedio en el sistema': avg_time_system,
        'Porcentaje de tiempo ocioso': pct_idle,
        'Tiempo de espera promedio por cliente': avg_wait_per_customer,
        'Fraccion de clientes que espero': frac_wait,
        'Tiempo de espera promedio de quienes esperaron': avg_wait_of_waiters,
    }
    return df, medidas


### Simulación de una corrida — 20 clientes

Se ejecuta una primera corrida completa de la simulación, con 20 clientes, para revisar que la tabla se construya correctamente antes de pasar a repetir el proceso varias veces.


In [ ]:
df20, medidas20 = simular_cola(20, lam_poisson=10, n_binom=10, p_binom=0.40, seed=1)
df20


,Customer,Time Between Arrivals,Arrival Time,Service Time,Service Begins,Time Service Ends,Time in System,Idle Time,Time in Queue
0,1,-,0,0,0,0,0,0,0
1,2,11,11,4,11,15,4,11,0
2,3,7,18,2,18,20,2,3,0
3,4,9,27,4,27,31,4,7,0
4,5,5,32,5,32,37,5,1,0
5,6,10,42,3,42,45,3,5,0
6,7,10,52,3,52,55,3,7,0
7,8,8,60,3,60,63,3,5,0
8,9,4,64,7,64,71,7,1,0
9,10,13,77,5,77,82,5,6,0


In [ ]:
pd.Series(medidas20, name='Valor').to_frame()


,Valor
Tiempo promedio en el sistema,3.850000
Porcentaje de tiempo ocioso,0.557471
Tiempo de espera promedio por cliente,0.000000
Fraccion de clientes que espero,0.000000
Tiempo de espera promedio de quienes esperaron,0.000000


## Repetición de la simulación (10 corridas) — 20 clientes

Aqui se repite el mismo procedimiento diez veces, cambiando la semilla del generador de números aleatorios en cada corrida, tal como se hizo en la Práctica 1. La idea de repetir la simulación varias veces, en lugar de quedarse con una sola corrida, es poder observar la variabilidad de los resultados de una corrida a otra y, más adelante, calcular intervalos de confianza en vez de reportar un único número.


In [ ]:
def diez_corridas(n_clientes, lam_poisson, n_binom, p_binom, base_seed):
    filas = []
    for run in range(1, 11):
        _, m = simular_cola(n_clientes, lam_poisson, n_binom, p_binom, seed=base_seed + run)
        filas.append({'Run': run, **m})
    return pd.DataFrame(filas)

corridas_20 = diez_corridas(20, 10, 10, 0.40, base_seed=1000)
corridas_20


,Run,Tiempo promedio en el sistema,Porcentaje de tiempo ocioso,Tiempo de espera promedio por cliente,Fraccion de clientes que espero,Tiempo de espera promedio de quienes esperaron
0,1,4.200000,0.569231,0.000000,0.000000,0.000000
1,2,4.250000,0.582915,0.100000,0.050000,2.000000
2,3,3.750000,0.625000,0.000000,0.000000,0.000000
3,4,3.900000,0.610000,0.000000,0.000000,0.000000
4,5,3.550000,0.630208,0.000000,0.000000,0.000000
5,6,4.150000,0.559783,0.100000,0.050000,2.000000
6,7,3.900000,0.610000,0.000000,0.000000,0.000000
7,8,4.300000,0.577889,0.100000,0.050000,2.000000
8,9,3.300000,0.682692,0.000000,0.000000,0.000000
9,10,3.400000,0.605882,0.050000,0.050000,1.000000


## Intervalos de confianza (95 % y 99 %)

Se entiende que, como cada corrida de la simulación arroja un resultado distinto para cada medida de desempeño, no es suficiente con reportar un solo valor: es mejor calcular un intervalo de confianza que indique, con cierto nivel de certeza, entre qué valores se espera que esté el verdadero promedio de esa medida.

Para construir cada intervalo se sigue este procedimiento: primero se calcula el promedio de los diez resultados obtenidos en las diez corridas, y también su desviación estándar (una medida de qué tan dispersos están esos resultados). Luego, usando un valor de tabla que depende del nivel de confianza que se quiera (95% o 99%) y del número de corridas, se calcula una semiancho del intervalo. Finalmente, el intervalo de confianza se construye restando esa semiancho al promedio para obtener el límite inferior, y sumándola al promedio para obtener el límite superior. Entre más alto es el nivel de confianza exigido (por ejemplo 99% en vez de 95%), más ancho resulta el intervalo, porque se necesita más margen para tener mayor seguridad de que el valor real esté dentro de él.


In [ ]:
def intervalos_confianza(tabla_corridas, n_clientes_label):
    medidas = [c for c in tabla_corridas.columns if c != 'Run']
    n = len(tabla_corridas)
    filas = []
    for medida in medidas:
        datos = tabla_corridas[medida].values
        xbar = datos.mean()
        s = datos.std(ddof=1)
        for conf in [0.95, 0.99]:
            alpha = 1 - conf
            t_val = stats.t.ppf(1 - alpha/2, df=n-1)
            h = t_val * s / np.sqrt(n)
            filas.append({
                'Clientes': n_clientes_label,
                'Medida': medida,
                'Confianza': f"{int(conf*100)}%",
                'X_barra': xbar,
                'S': s,
                't_(n-1,1-a/2)': t_val,
                'h': h,
                'IC_inf': xbar - h,
                'IC_sup': xbar + h,
            })
    return pd.DataFrame(filas)

ic_20 = intervalos_confianza(corridas_20, 20)
ic_20


,Clientes,Medida,Confianza,X_barra,S,"t_(n-1,1-a/2)",h,IC_inf,IC_sup
0,20,Tiempo promedio en el sistema,95%,3.870000,0.362246,2.262157,0.259135,3.610865,4.129135
1,20,Tiempo promedio en el sistema,99%,3.870000,0.362246,3.249836,0.372276,3.497724,4.242276
2,20,Porcentaje de tiempo ocioso,95%,0.605360,0.036006,2.262157,0.025757,0.579603,0.631117
3,20,Porcentaje de tiempo ocioso,99%,0.605360,0.036006,3.249836,0.037003,0.568357,0.642363
4,20,Tiempo de espera promedio por cliente,95%,0.035000,0.047434,2.262157,0.033932,0.001068,0.068932
5,20,Tiempo de espera promedio por cliente,99%,0.035000,0.047434,3.249836,0.048748,-0.013748,0.083748
6,20,Fraccion de clientes que espero,95%,0.020000,0.025820,2.262157,0.018470,0.001530,0.038470
7,20,Fraccion de clientes que espero,99%,0.020000,0.025820,3.249836,0.026535,-0.006535,0.046535
8,20,Tiempo de espera promedio de quienes esperaron,95%,0.700000,0.948683,2.262157,0.678647,0.021353,1.378647
9,20,Tiempo de espera promedio de quienes esperaron,99%,0.700000,0.948683,3.249836,0.974951,-0.274951,1.674951


## Comparación con los resultados de la Práctica 1 (Simulación Ad Hoc con dados/ruletas)

En la Práctica 1, los tiempos entre llegadas se generaron con una distribución uniforme discreta entre 1 y 10 (con un promedio de 5.5), y los tiempos de servicio con una distribución uniforme discreta entre 1 y 6 (con un promedio de 3.5). En esta práctica, en cambio, se usan una Poisson con parámetro 10 (promedio 10) para los tiempos entre llegadas, y una Binomial con n igual a 10 y p igual a 0.4 (promedio 4) para los tiempos de servicio.


In [ ]:
resumen_lab1_20 = {
    'Tiempo promedio en el sistema': 4.470,
    'Porcentaje de tiempo ocioso': 0.358,
    'Tiempo de espera promedio por cliente': 0.930,
    'Fraccion de clientes que espero': 0.295,
    'Tiempo de espera promedio de quienes esperaron': 3.006,
}

comparacion = pd.DataFrame({
    'Practica 1 (20 clientes, uniforme)': resumen_lab1_20,
    'Practica 2 (20 clientes, Poisson/Binomial)': corridas_20.drop(columns='Run').mean(),
})
comparacion


,"Practica 1 (20 clientes, uniforme)","Practica 2 (20 clientes, Poisson/Binomial)"
Tiempo promedio en el sistema,4.470000,3.870000
Porcentaje de tiempo ocioso,0.358000,0.605360
Tiempo de espera promedio por cliente,0.930000,0.035000
Fraccion de clientes que espero,0.295000,0.020000
Tiempo de espera promedio de quienes esperaron,3.006000,0.700000


### ¿Qué puede decirse de los resultados obtenidos? ¿Qué similitudes o diferencias se presentan frente a la Práctica 1?

**Similitudes:**
- Se observa que la relación entre las cinco medidas de desempeño se mantiene igual en las dos prácticas: el tiempo de espera promedio de los clientes que sí tuvieron que esperar sigue siendo notablemente mayor que el tiempo de espera promedio general (que incluye también a los clientes que no esperaron nada), y un nivel de confianza más alto (99% en vez de 95%) sigue produciendo intervalos más anchos.
- En ambos casos existe variabilidad de una corrida a otra, lo que confirma que es necesario repetir la simulación varias veces y reportar intervalos de confianza, en lugar de confiar en el resultado de una sola corrida.

**Diferencias:**
- El tiempo entre llegadas con distribución Poisson tiene un promedio de 10, casi el doble que el promedio de 5.5 de la uniforme discreta usada en la Práctica 1. Esto hace que, en esta práctica, los clientes lleguen en promedio más espaciados en el tiempo.
- El tiempo de servicio con distribución Binomial tiene un promedio de 4, un poco superior al promedio de 3.5 de la uniforme usada en la Práctica 1, y además su variabilidad es distinta: la Binomial tiene una varianza de 2.4, mientras que la uniforme discreta entre 1 y 6 tiene una varianza cercana a 2.92.
- Como consecuencia de que los clientes llegan más espaciados, el estudiante espera encontrar un mayor porcentaje de tiempo ocioso del servidor en esta práctica, y una fracción menor de clientes que deben esperar, ya que el servidor tiene más probabilidad de estar libre cuando llega cada nuevo cliente. Esto se puede confirmar comparando las columnas de la tabla de comparación construida anteriormente.
- Además, como la Poisson y la Binomial tienen formas distintas a la uniforme discreta (la Binomial queda más concentrada alrededor de su promedio y con colas más cortas, mientras que la Poisson tiene una cola derecha más larga que la uniforme), la forma en que se distribuyen los resultados de cada medida de desempeño entre las diez corridas también puede cambiar, lo cual afecta la desviación estándar muestral y, en consecuencia, el ancho de los intervalos de confianza obtenidos.
